# 1.准备数据

本代码在H800A（显存80G）上运行的结果，其他GPU根据显存可适当调整batch_size

In [31]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms
from PIL import Image
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, fn, taskType=0, transforms=None):
        f = open(fn,'r',encoding='GBK')
        ls = f.readlines()
        self.images = []
        self.labels = []
        for l in ls:
            l=l.strip()
            if len(l.strip())==0:continue
            fs = l.split(' ')
            self.images.append(fs[0])
            self.labels.append(int(fs[1]))
        self.transforms = transforms  # 对图像做变换

    def __getitem__(self, idx):
        # 得到第idx个图像文件名和掩码文件名
        img_path= self.images[idx]    
        img = Image.open(img_path).convert("RGB")  # 打开原始图像，并转换为RGB格式   
        if self.transforms is not None:
            img = self.transforms(img)
        return img_path, img, self.labels[idx]

    def __len__(self):
        return len(self.images)

    
original_dict={'footballField': 0, 'Farmland': 1, 'Beach': 2, 'Parking': 3, 'Airport': 4, 'Forest': 5, 'Park': 6, 'River': 7, 'Pond': 8, 'Desert': 9, 'Industrial': 10, 'Viaduct': 11, 'railwayStation': 12, 'Mountain': 13, 'Bridge': 14, 'Port': 15, 'Meadow': 16, 'Residential': 17, 'Commercial': 18}
name = {v: k for k, v in original_dict.items()}



def random_rot90(img):
    """对单张图像 (C, H, W) 随机旋转 0/90/180/270 度"""
    k = torch.randint(0, 4, ()).item()  # 0,1,2,3
    return torch.rot90(img, k, dims=[-2, -1])  # 在最后两个维度（H,W）上旋转

size = 224

# 定义图像预处理的 transforms
transform = transforms.Compose([
    transforms.Resize((size, size)),  # 将图像调整    
    transforms.ToTensor(),          # 将 PIL 图像或 numpy 数组转换为张量
    transforms.Lambda(random_rot90),
])


training_data = MyDataset( 
    fn='sattrain.txt',  # 存储训练数据集的路径
    taskType=0,  # 指定训练数据集
    transforms=transform,  # 指定特征和标签转换为张量
)
val_data = MyDataset(
    fn='satval.txt',  # 存储测试数据集的路径
    taskType=1,  # 指定测试数据集
    transforms=transform,
)

batch_size = 64  # 设置一次输入网络的样本数量为64
# 实例化训练数据加载器
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
# 实例化测试数据加载器
val_dataloader = DataLoader(val_data, batch_size=batch_size)
# 显示训练接和验证集样本数量
print(training_data.__len__(),val_data.__len__())

904 101


# 2.编写训练函数和验证函数

In [48]:
from torchvision.models.mobilenet import mobilenet_v2
from torch.optim.lr_scheduler import StepLR
from torch.nn import CrossEntropyLoss
import time
def train(model, device, train_loader, optimizer, epoch):
    log_interval = 2
    loss_func = CrossEntropyLoss()
    model.train()
    begin=time.time()
    for batch_idx, (imgs, data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = loss_func(output, target)
        loss.backward()
        optimizer.step()
        if (batch_idx+1) % log_interval == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} ' \
            f'({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}, '\
            f'used time:{time.time()-begin:.0f} s')


def tst(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    loss_func = CrossEntropyLoss()
    begin=time.time()
    with torch.no_grad():
        for imgs, data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += loss_func(output, target)
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            n = pred.eq(target.view_as(pred)).sum().item()
            correct += n
           
    test_loss /= len(test_loader)

    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} '\
           f'({100. * correct / len(test_loader.dataset):.0f}%) used time:{time.time()-begin:.0f} s\n')
    return test_loss

# 3. 执行代码


In [49]:
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 10
learning_rate = 1.0
reduce_lr_gamma = 0.7

model = mobilenet_v2(pretrained=True)
model.classifier[1] = torch.nn.Linear(in_features=model.classifier[1].in_features, out_features=len(name))
model.to(device)
optimizer = optim.Adadelta(model.parameters(), lr=learning_rate)

scheduler = StepLR(optimizer, step_size=1, gamma=reduce_lr_gamma)
loss=100
for epoch in range(1, epochs + 1):
        train(model, device, train_dataloader, optimizer, epoch)
        test_loss = tst(model, device, val_dataloader)
        if loss > test_loss:
            loss = test_loss
            torch.save(model, "sat_mobilenet.pt")
        scheduler.step()

Train Epoch: 1 [64/904 (7%)]	Loss: 2.565737, used time:5 s
Train Epoch: 1 [192/904 (20%)]	Loss: 2.618313, used time:9 s
Train Epoch: 1 [320/904 (33%)]	Loss: 3.729695, used time:14 s
Train Epoch: 1 [448/904 (47%)]	Loss: 3.288827, used time:19 s
Train Epoch: 1 [576/904 (60%)]	Loss: 4.053065, used time:24 s
Train Epoch: 1 [704/904 (73%)]	Loss: 2.330233, used time:28 s
Train Epoch: 1 [832/904 (87%)]	Loss: 1.654072, used time:33 s

Test set: Average loss: 8.7448, Accuracy: 8/101 (8%) used time:4 s

Train Epoch: 2 [64/904 (7%)]	Loss: 2.629591, used time:5 s
Train Epoch: 2 [192/904 (20%)]	Loss: 1.649356, used time:9 s
Train Epoch: 2 [320/904 (33%)]	Loss: 1.074206, used time:14 s
Train Epoch: 2 [448/904 (47%)]	Loss: 0.656887, used time:19 s
Train Epoch: 2 [576/904 (60%)]	Loss: 0.870971, used time:24 s
Train Epoch: 2 [704/904 (73%)]	Loss: 0.722254, used time:29 s
Train Epoch: 2 [832/904 (87%)]	Loss: 0.670870, used time:34 s

Test set: Average loss: 1.2536, Accuracy: 63/101 (62%) used time:4 s



# 4.直接加载训练好的模型文件，对图像进行分类识别

In [50]:
model = torch.load('sat_mobilenet.pt', weights_only=False)
model.eval()
img_paths=[r'dataset/Parking/parking_14.jpg',
           r'dataset/Beach/beach-30.jpg',
           r'dataset/Park/Park_18.jpg',
           r'dataset/footballField/footballField_42.jpg',
           r'dataset/Pond/pond_30.jpg',
           r'dataset/Parking/parking_36.jpg',
           r'dataset/Desert/Desert_04.jpg',
           r'dataset/Airport/airport_12.jpg',
           r'dataset/Beach/beach-09.jpg',
           r'dataset/Viaduct/viaduct_28.jpg',
           r'dataset/Farmland/Farmland-35.jpg',
           r'dataset/footballField/footballField_37.jpg',
           r'dataset/Commercial/commercial_22.jpg']

new_transform = transforms.Compose(transform.transforms[:-1])
imgs = [new_transform(Image.open(img_path)) for img_path in img_paths]  # 打开原始图像，并转换为RGB格式   
imgs = torch.stack(imgs, dim=0).to(device)
pred = model(imgs)
max_index = pred.argmax(dim=1, keepdim=True)
print("显示文件名，识别类别，结果是否准确：")
for img , i in zip(img_paths, max_index):
   print(img, name[i.item()], name[i.item()] in img)

显示文件名，识别类别，结果是否准确：
dataset/Parking/parking_14.jpg Parking True
dataset/Beach/beach-30.jpg Beach True
dataset/Park/Park_18.jpg Park True
dataset/footballField/footballField_42.jpg footballField True
dataset/Pond/pond_30.jpg Pond True
dataset/Parking/parking_36.jpg Parking True
dataset/Desert/Desert_04.jpg Desert True
dataset/Airport/airport_12.jpg Airport True
dataset/Beach/beach-09.jpg Beach True
dataset/Viaduct/viaduct_28.jpg Viaduct True
dataset/Farmland/Farmland-35.jpg Farmland True
dataset/footballField/footballField_37.jpg footballField True
dataset/Commercial/commercial_22.jpg Commercial True
